In [1]:
# setup and imports

import json
import joblib
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

# Paths
BASE_DIR = Path(".")
MODELS_DIR = BASE_DIR / "models"
PROCESSED_DIR = Path("..") / "data" / "processed"

# Device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [2]:
# load test data

df_test = pd.read_csv(PROCESSED_DIR / "test.csv")
mapping_df = pd.read_csv(PROCESSED_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping_df["label"], mapping_df["supercategory"]))
df_test["supercategory"] = df_test["label"].map(label_to_supercat)

print(f"Test samples: {len(df_test)}")
print(f"Cities: {df_test['city_group'].nunique()}")

Test samples: 5510
Cities: 41


In [3]:
# fairness helper functions

def gap_by_group(df, group_col):
    """Accuracy gap: max - min across groups."""
    group_acc = df.groupby(group_col)["correct"].mean().dropna()
    if len(group_acc) == 0:
        return np.nan
    return group_acc.max() - group_acc.min()


def ovr_rates_by_group(df, group_col, num_classes):
    """Compute one-vs-rest TPR and FPR for each class and group."""
    groups = sorted(df[group_col].dropna().unique())
    
    tpr = np.zeros((len(groups), num_classes))
    fpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    
    for gi, g in enumerate(groups):
        df_g = df[df[group_col] == g]
        y_true = df_g["y_true"].values
        y_pred = df_g["y_pred"].values
        
        for c in range(num_classes):
            pos_mask = (y_true == c)
            neg_mask = (y_true != c)
            
            TP = np.sum((y_pred == c) & pos_mask)
            FP = np.sum((y_pred == c) & neg_mask)
            FN = np.sum((y_pred != c) & pos_mask)
            TN = np.sum((y_pred != c) & neg_mask)
            
            support[gi, c] = np.sum(pos_mask)
            tpr[gi, c] = TP / (TP + FN) if (TP + FN) > 0 else np.nan
            fpr[gi, c] = FP / (FP + TN) if (FP + TN) > 0 else np.nan
    
    return tpr, fpr, support, groups


def summarize_gaps(rate_matrix, support_matrix=None, min_support=0):
    """Compute per-class gap, worst gap, macro gap. Optionally filter by support."""
    gap_per_class = []
    
    for c in range(rate_matrix.shape[1]):
        col = rate_matrix[:, c]
        
        # Filter by support if provided
        if support_matrix is not None and min_support > 0:
            mask = support_matrix[:, c] >= min_support
            col = col[mask]
        
        col = col[~np.isnan(col)]
        
        if len(col) < 2:  # Need at least 2 groups to compute gap
            gap_per_class.append(np.nan)
        else:
            gap_per_class.append(np.max(col) - np.min(col))
    
    gap_per_class = np.array(gap_per_class)
    valid = gap_per_class[~np.isnan(gap_per_class)]
    
    if len(valid) == 0:
        return gap_per_class, np.nan, np.nan
    
    return gap_per_class, np.max(valid), np.mean(valid)


def evaluate_fairness(df_eval, num_classes, min_support=30):
    """Full fairness evaluation with robust filtering."""
    results = {}
    
    # Basic metrics
    results["acc"] = accuracy_score(df_eval["y_true"], df_eval["y_pred"])
    results["macro_f1"] = f1_score(df_eval["y_true"], df_eval["y_pred"], average="macro")
    
    # Accuracy gaps
    df_eval["correct"] = df_eval["y_true"] == df_eval["y_pred"]
    results["gap_acc_city"] = gap_by_group(df_eval, "city_group")
    results["gap_acc_gender"] = gap_by_group(df_eval, "gender")
    
    # TPR/FPR gaps (full)
    tpr, fpr, support, groups = ovr_rates_by_group(df_eval, "city_group", num_classes)
    _, tpr_worst, tpr_macro = summarize_gaps(tpr)
    _, fpr_worst, fpr_macro = summarize_gaps(fpr)
    
    results["tpr_gap_worst"] = tpr_worst
    results["tpr_gap_macro"] = tpr_macro
    results["fpr_gap_worst"] = fpr_worst
    results["fpr_gap_macro"] = fpr_macro
    
    # TPR/FPR gaps (robust - filtered by support)
    _, tpr_worst_r, tpr_macro_r = summarize_gaps(tpr, support, min_support)
    _, fpr_worst_r, fpr_macro_r = summarize_gaps(fpr, support, min_support)
    
    results["tpr_gap_worst_robust"] = tpr_worst_r
    results["tpr_gap_macro_robust"] = tpr_macro_r
    results["fpr_gap_worst_robust"] = fpr_worst_r
    results["fpr_gap_macro_robust"] = fpr_macro_r
    results["min_support"] = min_support
    
    return results

print("Fairness functions defined.")

Fairness functions defined.


In [4]:
# model loading and inference function

def load_and_predict(model_path, df_test, batch_size=32):
    """Load a saved model and run inference on test set."""
    model_path = Path(model_path)
    
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path), local_files_only=True)
    model = model.to(device)
    model.eval()
    
    # Load label encoder
    le = joblib.load(model_path / "label_encoder.joblib")
    num_labels = len(le.classes_)
    
    # Prepare data
    df = df_test.copy()
    df["y_true"] = le.transform(df["supercategory"])
    
    def tokenize_fn(batch):
        return tokenizer(batch["resume_text"], padding="max_length", truncation=True, max_length=128)
    
    dataset = Dataset.from_pandas(df[["resume_text", "y_true"]])
    dataset = dataset.map(tokenize_fn, batched=True)
    dataset.set_format("torch", columns=["input_ids", "attention_mask", "y_true"])
    
    dataloader = DataLoader(dataset, batch_size=batch_size)
    all_preds = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Inference {model_path.name}", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
    
    df["y_pred"] = all_preds
    
    # Cleanup
    del model
    torch.mps.empty_cache() if device.type == "mps" else None
    
    return df, num_labels, le

print("Model loading function defined.")

Model loading function defined.


In [5]:
# list available models

print("Available models in models/ directory:")
for p in sorted(MODELS_DIR.iterdir()):
    if p.is_dir() and (p / "config.json").exists():
        print(f"  ✓ {p.name}")
    elif p.is_dir():
        print(f"  ? {p.name} (no config.json)")

Available models in models/ directory:
  ✓ bert_9classes_final
  ✓ bert_9classes_gdro_eta0.1_1ep
  ✓ bert_9classes_old_gdro_eta0.1_1ep
  ✓ bert_9classes_old_rw_1ep
  ? minilm_paraphrase_L3v2_lr_svm (no config.json)
  ? tfidf_logreg_binary_it (no config.json)


In [6]:
# define models to compare

MODELS = {
    "bert_9classes_final": "BERT 2ep sqrt_rw (current best)",
    "bert_9classes_old_rw_1ep": "BERT 1ep inverse_rw",
    "bert_9classes_old_gdro_eta0.1_1ep": "BERT 1ep GroupDRO v1",
    "bert_9classes_gdro_eta0.1_1ep": "BERT 1ep GroupDRO v2",
}

# Check which models actually exist
available_models = {}
for model_name, desc in MODELS.items():
    path = MODELS_DIR / model_name
    if path.exists() and (path / "config.json").exists():
        available_models[model_name] = desc
        print(f"✓ {model_name}: {desc}")
    else:
        print(f"✗ {model_name}: NOT FOUND")

print(f"\nWill evaluate {len(available_models)} models.")

✓ bert_9classes_final: BERT 2ep sqrt_rw (current best)
✓ bert_9classes_old_rw_1ep: BERT 1ep inverse_rw
✓ bert_9classes_old_gdro_eta0.1_1ep: BERT 1ep GroupDRO v1
✓ bert_9classes_gdro_eta0.1_1ep: BERT 1ep GroupDRO v2

Will evaluate 4 models.


In [7]:
# run evaluation

MIN_SUPPORT = 30  # Only count groups with >= 30 samples per class

all_results = []

for model_name, desc in available_models.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {desc}")
    print(f"{'='*60}")
    
    model_path = MODELS_DIR / model_name
    
    # Load and predict
    df_eval, num_labels, le = load_and_predict(model_path, df_test)
    
    # Evaluate fairness
    metrics = evaluate_fairness(df_eval, num_labels, min_support=MIN_SUPPORT)
    metrics["model"] = model_name
    metrics["description"] = desc
    
    all_results.append(metrics)
    
    # Print summary
    print(f"\nAccuracy: {metrics['acc']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print(f"City Acc Gap: {metrics['gap_acc_city']:.4f}")
    print(f"TPR Gap (worst, all): {metrics['tpr_gap_worst']:.4f}")
    print(f"TPR Gap (macro, all): {metrics['tpr_gap_macro']:.4f}")
    print(f"TPR Gap (worst, support>={MIN_SUPPORT}): {metrics['tpr_gap_worst_robust']:.4f}")
    print(f"TPR Gap (macro, support>={MIN_SUPPORT}): {metrics['tpr_gap_macro_robust']:.4f}")

print("\n" + "="*60)
print("Evaluation complete!")


Evaluating: BERT 2ep sqrt_rw (current best)


Map: 100%|██████████| 5510/5510 [00:11<00:00, 482.09 examples/s]



Accuracy: 0.6085
Macro F1: 0.6206
City Acc Gap: 0.6000
TPR Gap (worst, all): 1.0000
TPR Gap (macro, all): 0.9630
TPR Gap (worst, support>=30): 0.3294
TPR Gap (macro, support>=30): 0.1155

Evaluating: BERT 1ep inverse_rw


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 891.36it/s, Materializing param=classifier.weight]                                       
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Map: 100%|██████████| 5510/5510 [00:09<00:00, 556.60 examples/s]



Accuracy: 0.5946
Macro F1: 0.6084
City Acc Gap: 0.6701
TPR Gap (worst, all): 1.0000
TPR Gap (macro, all): 1.0000
TPR Gap (worst, support>=30): 0.3588
TPR Gap (macro, support>=30): 0.1184

Evaluating: BERT 1ep GroupDRO v1


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1451.38it/s, Materializing param=classifier.weight]                                      
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Map: 100%|██████████| 5510/5510 [00:08<00:00, 672.84 examples/s]



Accuracy: 0.5358
Macro F1: 0.5411
City Acc Gap: 0.6066
TPR Gap (worst, all): 1.0000
TPR Gap (macro, all): 0.9778
TPR Gap (worst, support>=30): 0.3353
TPR Gap (macro, support>=30): 0.1222

Evaluating: BERT 1ep GroupDRO v2


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1311.33it/s, Materializing param=classifier.weight]                                      
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Map: 100%|██████████| 5510/5510 [00:09<00:00, 610.55 examples/s]
                                                                                          


Accuracy: 0.4662
Macro F1: 0.4508
City Acc Gap: 0.4834
TPR Gap (worst, all): 1.0000
TPR Gap (macro, all): 0.8889
TPR Gap (worst, support>=30): 0.4176
TPR Gap (macro, support>=30): 0.1485

Evaluation complete!


In [8]:
# results comparison table

results_df = pd.DataFrame(all_results)

# Reorder columns for readability
cols = [
    "description", "acc", "macro_f1", 
    "gap_acc_city", 
    "tpr_gap_worst", "tpr_gap_macro",
    "tpr_gap_worst_robust", "tpr_gap_macro_robust",
    "model"
]
results_df = results_df[[c for c in cols if c in results_df.columns]]

# Round for display
display_df = results_df.copy()
for col in display_df.columns:
    if display_df[col].dtype == float:
        display_df[col] = display_df[col].round(4)

display_df

,description,acc,macro_f1,gap_acc_city,tpr_gap_worst,tpr_gap_macro,tpr_gap_worst_robust,tpr_gap_macro_robust,model
0,BERT 2ep sqrt_rw (current best),0.6085,0.6206,0.6000,1.0,0.9630,0.3294,0.1155,bert_9classes_final
1,BERT 1ep inverse_rw,0.5946,0.6084,0.6701,1.0,1.0000,0.3588,0.1184,bert_9classes_old_rw_1ep
2,BERT 1ep GroupDRO v1,0.5358,0.5411,0.6066,1.0,0.9778,0.3353,0.1222,bert_9classes_old_gdro_eta0.1_1ep
3,BERT 1ep GroupDRO v2,0.4662,0.4508,0.4834,1.0,0.8889,0.4176,0.1485,bert_9classes_gdro_eta0.1_1ep


In [9]:
# save results

results_df.to_csv("../figures/model_fairness_comparison.csv", index=False)
print("Saved to ../figures/model_fairness_comparison.csv")

Saved to ../figures/model_fairness_comparison.csv


In [10]:
# detailed city analysis

# Use the current best model
best_model = "bert_9classes_final"
model_path = MODELS_DIR / best_model

df_eval, num_labels, le = load_and_predict(model_path, df_test)
df_eval["correct"] = df_eval["y_true"] == df_eval["y_pred"]

# City-level accuracy
city_stats = df_eval.groupby("city_group").agg(
    n=("correct", "count"),
    accuracy=("correct", "mean")
).round(4)

city_stats = city_stats.sort_values("accuracy", ascending=False)
print("City-level accuracy (sorted):")
city_stats

Map: 100%|██████████| 5510/5510 [00:14<00:00, 381.73 examples/s]


City-level accuracy (sorted):


,n,accuracy
city_group,,
Минск,31,1.0000
Алматы,67,0.9701
Барнаул,21,0.7143
Уфа,84,0.7024
Калининград,23,0.6957
Екатеринбург,89,0.6854
Симферополь,18,0.6667
Владивосток,21,0.6667
Иркутск,33,0.6667


In [11]:
# TPR by city and class

tpr, fpr, support, groups = ovr_rates_by_group(df_eval, "city_group", num_labels)

# Create DataFrame
class_names = le.classes_
tpr_df = pd.DataFrame(tpr, index=groups, columns=class_names)
support_df = pd.DataFrame(support, index=groups, columns=class_names)

print("TPR by City x Class:")
tpr_df.round(2)

TPR by City x Class:


,backend_general_dev,generic_it_ops,it_governance_leadership,project_product,sales_account,sysadmin_devops_network,tech_support_helpdesk,technical_specialized,web_frontend
Other,0.56,0.57,0.51,0.49,0.58,0.74,0.45,0.46,0.59
Алматы,1.00,0.87,1.00,1.00,1.00,1.00,1.00,1.00,1.00
Барнаул,0.50,0.75,1.00,0.00,0.00,1.00,0.50,1.00,1.00
Владивосток,NaN,0.75,1.00,0.50,NaN,0.50,0.00,0.83,NaN
Волгоград,0.50,1.00,1.00,1.00,0.00,1.00,0.33,0.20,NaN
Воронеж,0.57,0.59,0.43,0.43,0.43,0.93,0.29,0.75,1.00
Екатеринбург,0.67,0.84,0.67,0.56,0.50,0.58,0.20,0.93,0.67
Ижевск,0.50,0.50,0.50,0.00,0.50,0.50,NaN,1.00,NaN
Иркутск,1.00,1.00,0.00,0.67,NaN,0.73,0.33,0.40,NaN
Казань,0.33,0.63,0.70,0.54,0.00,0.64,0.50,0.50,1.00


In [12]:
# worst TPR gaps by class

MIN_SUPPORT = 30

gap_analysis = []

for c_idx, c_name in enumerate(class_names):
    # Filter cities with enough support for this class
    mask = support[:, c_idx] >= MIN_SUPPORT
    
    if mask.sum() < 2:
        continue
    
    valid_tpr = tpr[mask, c_idx]
    valid_groups = np.array(groups)[mask]
    valid_support = support[mask, c_idx]
    
    best_idx = np.nanargmax(valid_tpr)
    worst_idx = np.nanargmin(valid_tpr)
    
    gap_analysis.append({
        "class": c_name,
        "best_city": valid_groups[best_idx],
        "best_tpr": valid_tpr[best_idx],
        "best_support": int(valid_support[best_idx]),
        "worst_city": valid_groups[worst_idx],
        "worst_tpr": valid_tpr[worst_idx],
        "worst_support": int(valid_support[worst_idx]),
        "tpr_gap": valid_tpr[best_idx] - valid_tpr[worst_idx],
        "n_cities": int(mask.sum())
    })

gap_df = pd.DataFrame(gap_analysis)
gap_df = gap_df.sort_values("tpr_gap", ascending=False)

print(f"TPR Gap Analysis (cities with support >= {MIN_SUPPORT}):")
gap_df.round(3)

TPR Gap Analysis (cities with support >= 30):


,class,best_city,best_tpr,best_support,worst_city,worst_tpr,worst_support,tpr_gap,n_cities
0,backend_general_dev,Москва,0.771,170,Санкт-Петербург,0.441,34,0.329,3
1,generic_it_ops,Москва,0.703,239,Other,0.570,265,0.133,6
7,technical_specialized,Москва,0.589,209,Other,0.456,147,0.133,3
3,project_product,Санкт-Петербург,0.612,80,Other,0.489,131,0.124,3
2,it_governance_leadership,Москва,0.629,342,Other,0.510,96,0.118,3
4,sales_account,Other,0.584,77,Санкт-Петербург,0.473,55,0.112,3
8,web_frontend,Other,0.588,51,Москва,0.537,67,0.051,2
6,tech_support_helpdesk,Other,0.447,76,Санкт-Петербург,0.417,36,0.031,3
5,sysadmin_devops_network,Other,0.739,333,Санкт-Петербург,0.730,100,0.009,3


In [13]:
# save detailed analysid

gap_df.to_csv("../figures/tpr_gap_by_class_robust.csv", index=False)
city_stats.to_csv("../figures/city_accuracy_stats.csv")

print("Saved:")
print("  - ../figures/tpr_gap_by_class_robust.csv")
print("  - ../figures/city_accuracy_stats.csv")

Saved:
  - ../figures/tpr_gap_by_class_robust.csv
  - ../figures/city_accuracy_stats.csv


In [14]:
# summary for paper

print("="*60)
print("SUMMARY FOR PAPER")
print("="*60)

if len(all_results) > 0:
    best = all_results[0]  # Assuming first model is best
    
    print(f"\nBest Model: {best['description']}")
    print(f"  Accuracy: {best['acc']:.3f}")
    print(f"  Macro F1: {best['macro_f1']:.3f}")
    print(f"")
    print(f"Fairness Metrics (all cities):")
    print(f"  TPR Gap (worst): {best['tpr_gap_worst']:.3f}")
    print(f"  TPR Gap (macro): {best['tpr_gap_macro']:.3f}")
    print(f"")
    print(f"Fairness Metrics (robust, support >= {MIN_SUPPORT}):")
    print(f"  TPR Gap (worst): {best['tpr_gap_worst_robust']:.3f}")
    print(f"  TPR Gap (macro): {best['tpr_gap_macro_robust']:.3f}")
    print(f"")
    print("Note: Robust metrics exclude city-class combinations")
    print(f"with fewer than {MIN_SUPPORT} test samples to avoid")
    print("statistical noise from small groups.")

SUMMARY FOR PAPER

Best Model: BERT 2ep sqrt_rw (current best)
  Accuracy: 0.609
  Macro F1: 0.621

Fairness Metrics (all cities):
  TPR Gap (worst): 1.000
  TPR Gap (macro): 0.963

Fairness Metrics (robust, support >= 30):
  TPR Gap (worst): 0.329
  TPR Gap (macro): 0.116

Note: Robust metrics exclude city-class combinations
with fewer than 30 test samples to avoid
statistical noise from small groups.
